<a href="https://colab.research.google.com/github/VictoriaO26/AI-assisted-Cloud-Infrastructure-Cybersecurity-Bootcamp/blob/main/AI-assisted-Cloud-Infrastructure-Cybersecurity-Bootcamp/Module%203/Malware_Detection_notebook_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — AI-Assisted Malware Detection in Google Colab





**Objective**: The students will build a defensive machine learning detector that classifies cloud workload behavior as benign or malicious.

**Acknowledgement**: Acknowledgment: This module was developed under the National Science Foundation awards # 2416990, 2416992, and 2607393 at Tennessee Tech University, North Carolina A&T State University, and the University at Albany.







This notebook is the **student TODO version** of the lab. It is designed for Google Colab and contains `# TODO` lines that students must complete.

## Student task

Complete only the code lines marked with `# TODO`. The rest of the code is already prepared and should remain unchanged.

## What this notebook does

1. Downloads the raw Cuckoo Sandbox JSON reports folder from GitHub into `/content/Cuckoo_reports/`.
2. Extracts API-call features from `behavior → apistats`.
3. Sums API-call counts across all subprocesses in each report.
4. Builds a machine-learning table where each row is a sample and each column is an API call.
5. Adds the label column `Malware`, where `1 = malware` and `0 = benign`.
6. Sanitizes missing or invalid values.
7. Scales the feature matrix using `StandardScaler`.
8. Trains and compares multiple scikit-learn models.
9. Selects the best model based on test accuracy and balanced accuracy.
10. Saves the model, feature table, metrics, classification report, confusion matrix, and final submission ZIP.

## Important behavior

This notebook **does not automatically download anything to your browser**. It only writes files inside the Colab runtime under `/content/`. At the end, it prints the output paths so you can inspect or move the files manually.

## Input requirement

This version **does not contain embedded data**. The setup cell downloads the required `Cuckoo_reports` folder from GitHub before the data-loading cell runs.

### Option A — Full raw-report run, recommended

Run the folder-preparation cell. It creates this folder:

```text
/content/Cuckoo_reports/
```

with Cuckoo JSON reports from the GitHub repository.

Optional labels file:

```text
/content/labels.csv
```

with this format:

```csv
filename,label
sample001.json,benign
sample002.json,malware
```

When `labels.csv` is absent, the notebook infers labels from filenames. Filenames containing `benign`, `clean`, `goodware`, `normal`, or `safe` are labeled `0`; filenames containing malware-related words are labeled `1`; ambiguous filenames default to `1` with a warning.

### Option B — Existing extracted feature file

Upload this file:

```text
/content/extractedInfo.json
```

This skips raw-report extraction and starts from already-extracted API-call features.

## Output location

The final ZIP is created here. The ZIP intentionally excludes auxiliary files such as standalone scripts, `accuracy.txt`, `feature_names.json`, video-demo checklist, and package-list files:

```text
/content/output_output_lab3.zip
```

The intermediate outputs are created here:

```text
/content/malware_detection_outputs/
```

The unpacked submission folder is created here:

```text
/content/malware_detection_submission/
```


## 1. Imports and configuration

This cell imports the libraries used throughout the notebook and defines all paths and experiment settings in one place. The paths are written for Google Colab, where `/content/` is the default working area.


In [ ]:
# ============================================================
# 1. Imports and configuration
# ============================================================

from pathlib import Path
import json
import shutil
import subprocess
import sys
import warnings
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier

import joblib
import matplotlib.pyplot as plt

STUDENT_FIRST_NAME = "output"
STUDENT_LAST_NAME = "output"
SUBMISSION_ZIP_NAME = f"{STUDENT_LAST_NAME}_{STUDENT_FIRST_NAME}_lab3.zip"

REPORTS_DIR = Path("/content/Cuckoo_reports")
LABELS_CSV = Path("/content/labels.csv")
EXTRACTED_JSON = Path("/content/extractedInfo.json")

OUTPUT_DIR = Path("/content/malware_detection_outputs")
SUBMISSION_DIR = Path("/content/malware_detection_submission")
SUBMISSION_ZIP = Path("/content") / SUBMISSION_ZIP_NAME

RANDOM_STATE = 42
TEST_SIZE = 0.25

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"Reports folder: {REPORTS_DIR}")
print(f"Output folder:  {OUTPUT_DIR}")
print(f"Final ZIP:      {SUBMISSION_ZIP}")

## 2. Input files

The notebook prepares the raw-report input by downloading only the `Cuckoo_reports` folder from the GitHub repository.

After that, the data-loading cell checks inputs in this order:

1. Raw Cuckoo reports in `/content/Cuckoo_reports/`.
2. A pre-existing `/content/extractedInfo.json` file.

This notebook intentionally does **not** include embedded extracted data. That keeps the notebook cleaner and makes the run depend on explicit project input files.


In [ ]:
%%bash
set -e

REPO_URL="https://github.com/AI4SecureCI/AI4SecureCI.git"
FOLDER="Cuckoo_reports"
TMP_DIR="AI4SecureCI_tmp"

# Remove only previous download/temp copy
rm -rf "$TMP_DIR" "$FOLDER"

# Download only the required folder
git clone --depth 1 --filter=blob:none --sparse "$REPO_URL" "$TMP_DIR"

cd "$TMP_DIR"
git sparse-checkout set "$FOLDER"
cd ..

# Put Cuckoo_reports directly in the Colab working directory
mv "$TMP_DIR/$FOLDER" .

# Clean temporary cloned repository
rm -rf "$TMP_DIR"

echo "Ready:"
ls -lah "$FOLDER"


## 3. Feature extraction functions

These functions implement the feature-extraction requirement:

- Open each Cuckoo JSON report.
- Read `behavior["apistats"]`.
- Iterate through every subprocess entry.
- Sum API-call counts across subprocesses.
- Build a unified list of API-call columns.
- Create one row per report.
- Assign a malware/benign label.

The result matches the expected `extractedInfo.json` structure used by the training step.


In [ ]:
# ============================================================
# 3. Feature extraction functions
# ============================================================


def read_labels_csv(labels_csv: Path) -> dict:
    """Read optional labels.csv mapping report filenames to labels."""
    if not labels_csv.exists():
        return {}
    df = pd.read_csv(labels_csv)
    if not {'filename', 'label'}.issubset(df.columns):
        raise ValueError('labels.csv must contain columns: filename,label')
    label_map = {}
    for _, row in df.iterrows():
        filename = str(row['filename']).strip()
        raw_label = str(row['label']).strip().lower()
        if raw_label in {'0', 'benign', 'clean', 'good', 'safe'}:
            label = 0
        elif raw_label in {'1', 'malware', 'malicious', 'bad', 'infected'}:
            label = 1
        else:
            raise ValueError(f"Unknown label {row['label']} for file {filename}")
        label_map[filename] = label
    print(f"Loaded {len(label_map)} labels from {labels_csv}")
    return label_map


def infer_label_from_filename(report_path: Path, label_map: dict | None = None) -> int:
    """Infer malware/benign label from labels.csv or filename."""
    label_map = label_map or {}
    if report_path.name in label_map:
        return int(label_map[report_path.name])

    lower = str(report_path).lower()
    benign_tokens = ['benign', 'clean', 'goodware', 'normal', 'safe']
    malware_tokens = ['malware', 'malicious', 'virus', 'trojan', 'ransom', 'adware', 'spyware', 'keylog']
    if any(token in lower for token in benign_tokens):
        return 0
    if any(token in lower for token in malware_tokens):
        return 1

    warnings.warn(
        f"Could not infer label for {report_path.name}; defaulting to malware=1. "
        'Provide labels.csv for ambiguous filenames.'
    )
    return 1


def extract_apistats_from_report(report_path: Path) -> dict:
    """Extract and sum API-call counts from one Cuckoo JSON report.

    Cuckoo reports normally store API statistics at:
        behavior -> apistats -> process_id -> api_call_name -> count

    The lab requires summing apistats over all subprocesses.
    """
    with open(report_path, 'r', encoding='utf-8', errors='replace') as f:
        data = json.load(f)

    # TODO 1: Locate the API-call statistics dictionary from the report JSON.
    if not isinstance(apistats, dict) or not apistats:
        raise ValueError('Missing or empty behavior.apistats')

    api_counts = {}
    for process_id, process_calls in apistats.items():
        if not isinstance(process_calls, dict):
            continue
        for api_name, count in process_calls.items():
            try:
                numeric_count = float(count)
            except (TypeError, ValueError):
                numeric_count = 0.0
            api_name = str(api_name)
            # TODO 2: Sum this API-call count into the sample's API-count dictionary.

    if not api_counts:
        raise ValueError('No usable API calls found in behavior.apistats')
    return api_counts


def extract_features_from_cuckoo_reports(reports_dir: Path, labels_csv: Path | None = None) -> dict:
    """Extract API-call feature vectors from all Cuckoo JSON reports in a folder."""
    if not reports_dir.exists():
        raise FileNotFoundError(f"Reports directory not found: {reports_dir}")

    label_map = read_labels_csv(labels_csv) if labels_csv else {}
    report_files = sorted(reports_dir.rglob('*.json'))
    report_files = [p for p in report_files if p.name != 'extractedInfo.json']
    if not report_files:
        raise FileNotFoundError(f"No .json Cuckoo reports found in {reports_dir}")

    all_file_dicts = []
    malware_labels = []
    sample_files = []
    all_api_names = []
    skipped = []

    for report_path in report_files:
        try:
            api_counts = extract_apistats_from_report(report_path)
        except Exception as exc:
            skipped.append((str(report_path), str(exc)))
            continue
        all_file_dicts.append(api_counts)
        malware_labels.append(infer_label_from_filename(report_path, label_map))
        sample_files.append(str(report_path.relative_to(reports_dir)))
        all_api_names.extend(api_counts.keys())

    if not all_file_dicts:
        example_errors = '\n'.join([f"- {p}: {e}" for p, e in skipped[:10]])
        raise RuntimeError('No usable Cuckoo reports were parsed. First errors:\n' + example_errors)

    # TODO 3: Build the complete shared list of API-call feature names.
    # TODO 4: Convert each sample dictionary into a numeric feature row.

    extracted_data = {
        'completeCallList': complete_call_list,
        'allFilesList': all_file_dicts,
        'finishedRows': finished_rows,
        'malwareList': malware_labels,
        'sampleFiles': sample_files,
        'skippedReports': skipped,
    }

    print(f"Parsed usable reports: {len(all_file_dicts)}")
    print(f"Skipped reports:       {len(skipped)}")
    print(f"API-call features:     {len(complete_call_list)}")
    print(f"Benign samples:        {sum(1 for y in malware_labels if y == 0)}")
    print(f"Malware samples:       {sum(1 for y in malware_labels if y == 1)}")
    return extracted_data

## 4. Load data

This cell chooses the best available input source. Raw reports are preferred because they demonstrate the full pipeline from Cuckoo JSON reports to model training. If raw reports are not present, the notebook uses `/content/extractedInfo.json`.

Because the embedded fallback dataset was removed, the notebook will stop with a clear error if neither input source is available.


In [ ]:
# ============================================================
# 4. Load data: raw Cuckoo reports -> extractedInfo.json
# ============================================================

source_description = None

if REPORTS_DIR.exists() and any(REPORTS_DIR.rglob('*.json')):
    print('Using raw Cuckoo reports as input.')
    extracted_data = extract_features_from_cuckoo_reports(
        reports_dir=REPORTS_DIR,
        labels_csv=LABELS_CSV if LABELS_CSV.exists() else None,
    )
    source_description = f'raw Cuckoo reports from {REPORTS_DIR}'
elif EXTRACTED_JSON.exists():
    print(f'Using existing extracted feature file: {EXTRACTED_JSON}')
    with open(EXTRACTED_JSON, 'r', encoding='utf-8') as f:
        extracted_data = json.load(f)
    source_description = f'existing extractedInfo.json at {EXTRACTED_JSON}'
else:
    raise FileNotFoundError(
        'No input data found. Run the folder-preparation cell to create /content/Cuckoo_reports/ with JSON reports, '
        'or upload /content/extractedInfo.json, then run the notebook again.'
    )

with open(OUTPUT_DIR / 'extractedInfo.json', 'w', encoding='utf-8') as f:
    json.dump(extracted_data, f, indent=2)

print(f"Data source: {source_description}")
print(f"Saved normalized extracted data to: {OUTPUT_DIR / 'extractedInfo.json'}")


## 5. Validate and format the dataset

This cell converts the extracted data into the machine-learning table required by the lab:

- `X`: API-call feature matrix.
- `y`: malware label vector.
- `feature_table.csv`: final table with API-call columns and the `Malware` label column.

It also handles missing values and saves a dataset summary.


In [ ]:
# ============================================================
# 5. Validate and format the extracted data for machine learning
# ============================================================

required_keys = {'completeCallList', 'finishedRows', 'malwareList'}
missing_keys = required_keys - set(extracted_data.keys())
if missing_keys:
    raise KeyError(f'extracted_data is missing required keys: {missing_keys}')

api_columns = [str(c) for c in extracted_data['completeCallList']]
# TODO 5: Separate the feature rows and Malware labels into X_raw and y.

if len(X_raw) != len(y):
    raise ValueError(f'Feature rows ({len(X_raw)}) and labels ({len(y)}) do not match.')

X = X_raw.apply(pd.to_numeric, errors='coerce')
X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

feature_table = X.copy()
feature_table['Malware'] = y.values
feature_table.to_csv(OUTPUT_DIR / 'feature_table.csv', index=False)

summary = {
    'source': source_description,
    'samples': int(X.shape[0]),
    'features': int(X.shape[1]),
    'benign_samples': int((y == 0).sum()),
    'malware_samples': int((y == 1).sum()),
    'created_at': datetime.utcnow().isoformat() + 'Z',
}
with open(OUTPUT_DIR / 'data_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Formatted feature table:')
print(f'  Samples:       {X.shape[0]}')
print(f'  API features:  {X.shape[1]}')
print(f'  Benign:        {(y == 0).sum()}')
print(f'  Malware:       {(y == 1).sum()}')
print(f"  Saved table:   {OUTPUT_DIR / 'feature_table.csv'}")

feature_table.head()

## 6. Train/test split and model setup

This cell splits the dataset into training and testing sets using stratification so both benign and malware samples are represented. Each candidate model is placed inside a pipeline with `StandardScaler`, satisfying the scaling requirement before model training.


In [ ]:
# ============================================================
# 6. Train/test split and model definitions
# ============================================================

if y.nunique() < 2:
    raise ValueError('The dataset must contain at least two classes: benign and malware.')

# TODO 6: Create the train/test split with the configured test size, random state, and stratification.

print('Train/test split complete.')
print(f'  Training samples: {len(X_train)}')
print(f'  Testing samples:  {len(X_test)}')

# A small, fast set of models is enough for the lab and keeps the notebook easy to run.
# Every pipeline includes StandardScaler, satisfying the scaling requirement.
candidate_models = {
    'ExtraTrees': ExtraTreesClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced'),
    'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    'LogisticRegression': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE),
}

# TODO 7: Build the model pipeline(s) using StandardScaler and the classifier.

print(f'Prepared {len(pipelines)} candidate models. Every pipeline includes StandardScaler.')

## 7. Train, compare, and select the best model

This cell trains several common scikit-learn classifiers, evaluates them on the held-out test set, and chooses the best model using accuracy and balanced accuracy. Balanced accuracy is included because malware datasets can be class-imbalanced.


In [ ]:
# ============================================================
# 7. Train, compare, and select the best model
# ============================================================

results = []
trained_models = {}

for name, pipeline in pipelines.items():
    model = clone(pipeline)
    # TODO 8: Train the model on the training data.
    # TODO 9: Predict the labels for the test data.

    # TODO 10: Compute the test accuracy.
    bal_acc = balanced_accuracy_score(y_test, y_pred_tmp)

    results.append({
        'model': name,
        'test_accuracy': float(acc),
        'test_balanced_accuracy': float(bal_acc),
    })
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(
    by=['test_accuracy', 'test_balanced_accuracy'], ascending=False
).reset_index(drop=True)

best_model_name = results_df.loc[0, 'model']
best_model = trained_models[best_model_name]
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
report_text = classification_report(y_test, y_pred, target_names=['Benign', 'Malware'])
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

print('Model comparison:')
display(results_df)
print(f'\nSelected model: {best_model_name}')
print(f'Accuracy: {accuracy:.6f}')
print(f'Balanced accuracy: {balanced_accuracy:.6f}')
print('\nClassification report:')
print(report_text)
print('Confusion matrix rows=true, columns=predicted, labels=[Benign, Malware]:')
print(cm)

## 8. Save model, metrics, reports, and plots

This cell saves the trained model and all evaluation artifacts to `/content/malware_detection_outputs/`. These files are also copied into the final submission ZIP later.


In [ ]:
# ============================================================
# 8. Save model, metrics, reports, and plots
# ============================================================

model_path = OUTPUT_DIR / 'model.joblib'
# TODO 11: Save the trained model pipeline to model_path.

with open(OUTPUT_DIR / 'feature_names.json', 'w', encoding='utf-8') as f:
    json.dump(api_columns, f, indent=2)

results_df.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

with open(OUTPUT_DIR / 'classification_report.txt', 'w', encoding='utf-8') as f:
    f.write(report_text)

pd.DataFrame(cm, index=['true_benign', 'true_malware'], columns=['predicted_benign', 'predicted_malware']).to_csv(
    OUTPUT_DIR / 'confusion_matrix.csv'
)

metrics = {
    'selected_model': best_model_name,
    'accuracy': float(accuracy),
    'balanced_accuracy': float(balanced_accuracy),
    'test_size': TEST_SIZE,
    'random_state': RANDOM_STATE,
    'samples': int(X.shape[0]),
    'features': int(X.shape[1]),
    'benign_samples': int((y == 0).sum()),
    'malware_samples': int((y == 1).sum()),
    'source': source_description,
    'created_at': datetime.utcnow().isoformat() + 'Z',
}
with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

with open(OUTPUT_DIR / 'accuracy.txt', 'w', encoding='utf-8') as f:
    f.write(f'{accuracy:.6f}\n')

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm)
ax.set_title(f'Confusion Matrix — {best_model_name}')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Benign', 'Malware'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['Benign', 'Malware'])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center')
fig.tight_layout()
plot_path = OUTPUT_DIR / 'confusion_matrix.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print('Saved outputs:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'  - {path}')

## 9. Generate auxiliary deliverables outside the final ZIP

This cell may generate standalone helper scripts and auxiliary documentation inside `/content/malware_detection_outputs/` for inspection. The notebook remains the main high-level file.

Per your packaging preference, the final ZIP built later will **not** include these auxiliary files:

- `extract_features.py`
- `train_model.py`
- `accuracy.py`
- `accuracy.txt`
- `feature_names.json`
- `VIDEO_DEMO_CHECKLIST.md`
- `package_list.txt`
- `full package_list.txt`



In [ ]:
# ============================================================
# 9. Generate deliverables and package list from inside the notebook
# ============================================================
# The notebook is the only source file you need to run.
# This cell creates the scripts and files requested by the lab inside the final ZIP.

extract_script = '"""extract_features.py\n\nExtract API-call features from Cuckoo Sandbox JSON reports.\n\nUsage:\n    python extract_features.py --reports Cuckoo_reports --output extractedInfo.json\n\nThis script reads behavior.apistats from each report and sums API-call counts\nacross all subprocesses in the report.\n"""\nfrom pathlib import Path\nimport argparse\nimport json\nimport warnings\n\n\ndef infer_label_from_filename(report_path: Path) -> int:\n    lower = str(report_path).lower()\n    if any(token in lower for token in ["benign", "clean", "goodware", "normal", "safe"]):\n        return 0\n    if any(token in lower for token in ["malware", "malicious", "virus", "trojan", "ransom", "adware", "spyware", "keylog"]):\n        return 1\n    warnings.warn(f"Could not infer label for {report_path.name}; defaulting to malware=1.")\n    return 1\n\n\ndef extract_apistats_from_report(report_path: Path) -> dict:\n    with open(report_path, "r", encoding="utf-8", errors="replace") as f:\n        data = json.load(f)\n    apistats = data.get("behavior", {}).get("apistats", {})\n    if not isinstance(apistats, dict) or not apistats:\n        raise ValueError("Missing or empty behavior.apistats")\n    api_counts = {}\n    for process_calls in apistats.values():\n        if not isinstance(process_calls, dict):\n            continue\n        for api_name, count in process_calls.items():\n            try:\n                numeric_count = float(count)\n            except (TypeError, ValueError):\n                numeric_count = 0.0\n            api_counts[str(api_name)] = api_counts.get(str(api_name), 0.0) + numeric_count\n    if not api_counts:\n        raise ValueError("No usable API calls found")\n    return api_counts\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--reports", default="Cuckoo_reports", help="Folder containing Cuckoo JSON reports")\n    parser.add_argument("--output", default="extractedInfo.json", help="Output JSON file")\n    args = parser.parse_args()\n\n    reports_dir = Path(args.reports)\n    report_files = sorted(p for p in reports_dir.rglob("*.json") if p.name != "extractedInfo.json")\n    if not report_files:\n        raise FileNotFoundError(f"No JSON reports found in {reports_dir}")\n\n    all_dicts, labels, sample_files, all_apis, skipped = [], [], [], [], []\n    for path in report_files:\n        try:\n            counts = extract_apistats_from_report(path)\n        except Exception as exc:\n            skipped.append((str(path), str(exc)))\n            continue\n        all_dicts.append(counts)\n        labels.append(infer_label_from_filename(path))\n        sample_files.append(str(path.relative_to(reports_dir)))\n        all_apis.extend(counts.keys())\n\n    if not all_dicts:\n        raise RuntimeError("No usable reports parsed.")\n\n    complete = list(dict.fromkeys(all_apis))\n    rows = [[d.get(api, 0.0) for api in complete] for d in all_dicts]\n    extracted = {\n        "completeCallList": complete,\n        "allFilesList": all_dicts,\n        "finishedRows": rows,\n        "malwareList": labels,\n        "sampleFiles": sample_files,\n        "skippedReports": skipped,\n    }\n    with open(args.output, "w", encoding="utf-8") as f:\n        json.dump(extracted, f, indent=2)\n    print(f"Wrote {args.output}")\n    print(f"Samples: {len(rows)}, API features: {len(complete)}, skipped: {len(skipped)}")\n\n\nif __name__ == "__main__":\n    main()\n'
train_script = '"""train_model.py\n\nTrain a malware-detection model from extractedInfo.json.\n\nUsage:\n    python train_model.py --input extractedInfo.json --output-dir outputs\n"""\nfrom pathlib import Path\nimport argparse\nimport json\nimport numpy as np\nimport pandas as pd\nimport joblib\nfrom sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", default="extractedInfo.json")\n    parser.add_argument("--output-dir", default="outputs")\n    args = parser.parse_args()\n\n    out = Path(args.output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n\n    with open(args.input, "r", encoding="utf-8") as f:\n        data = json.load(f)\n\n    columns = [str(c) for c in data["completeCallList"]]\n    X = pd.DataFrame(data["finishedRows"], columns=columns).apply(pd.to_numeric, errors="coerce")\n    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)\n    y = pd.Series(data["malwareList"]).astype(int)\n\n    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)\n    candidates = {\n        "ExtraTrees": ExtraTreesClassifier(n_estimators=400, random_state=42, class_weight="balanced"),\n        "RandomForest": RandomForestClassifier(n_estimators=400, random_state=42, class_weight="balanced"),\n    }\n    best_name, best_model, best_accuracy = None, None, -1\n    for name, clf in candidates.items():\n        pipe = Pipeline([("scaler", StandardScaler()), ("model", clf)])\n        pipe.fit(X_train, y_train)\n        pred = pipe.predict(X_test)\n        acc = accuracy_score(y_test, pred)\n        if acc > best_accuracy:\n            best_name, best_model, best_accuracy = name, pipe, acc\n\n    pred = best_model.predict(X_test)\n    cm = confusion_matrix(y_test, pred, labels=[0, 1])\n    report = classification_report(y_test, pred, target_names=["Benign", "Malware"])\n    joblib.dump(best_model, out / "model.joblib")\n    with open(out / "accuracy.txt", "w", encoding="utf-8") as f:\n        f.write(f"{best_accuracy:.6f}\\n")\n    with open(out / "classification_report.txt", "w", encoding="utf-8") as f:\n        f.write(report)\n    pd.DataFrame(cm).to_csv(out / "confusion_matrix.csv", index=False)\n    with open(out / "metrics.json", "w", encoding="utf-8") as f:\n        json.dump({\n            "selected_model": best_name,\n            "accuracy": float(best_accuracy),\n            "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),\n            "samples": int(X.shape[0]),\n            "features": int(X.shape[1]),\n        }, f, indent=2)\n    print(f"Selected model: {best_name}")\n    print(f"Accuracy: {best_accuracy:.6f}")\n\n\nif __name__ == "__main__":\n    main()\n'
accuracy_script = '"""accuracy.py\n\nPrint only the model accuracy as a number out of 1.\n\nUsage:\n    python accuracy.py\n"""\nfrom pathlib import Path\naccuracy_file = Path("outputs/accuracy.txt")\nif not accuracy_file.exists():\n    accuracy_file = Path("accuracy.txt")\nprint(accuracy_file.read_text(encoding="utf-8").strip())\n'

readme_text = f"""# Lab 3 - AI-Assisted Malware Detection

Student: {STUDENT_FIRST_NAME} {STUDENT_LAST_NAME}

This submission was generated by a single all-in-one Google Colab notebook.

## Workflow

1. Extract API-call features from Cuckoo Sandbox reports using `behavior.apistats`.
2. Sum API-call counts across all subprocesses in each report.
3. Format samples as rows and API calls as columns.
4. Add the `Malware` label column, where `1 = malware` and `0 = benign`.
5. Sanitize and standardize the data.
6. Train and compare machine-learning models.
7. Save the trained model and print accuracy as a number out of 1.

## Important files

- `extract_features.py`: feature extraction script.
- `train_model.py`: model training script.
- `accuracy.py`: prints the saved accuracy only.
- `outputs/model.joblib`: trained model pipeline.
- `outputs/accuracy.txt`: saved accuracy.
- `outputs/classification_report.txt`: precision, recall, F1, and accuracy.
- `outputs/confusion_matrix.csv`: confusion matrix.
- `package_list.txt`: package list.

## Accuracy

Selected model: {best_model_name}
Accuracy: {accuracy:.6f}
"""

video_checklist = f"""# Video Demo Checklist

1. Open the Colab notebook.
2. Show the input section: raw Cuckoo reports or `/content/extractedInfo.json`.
3. Run the feature extraction/formatting cells.
4. Show the feature table with samples as rows and API calls as columns.
5. Show the label distribution: benign vs malware.
6. Run the training cells.
7. Show the model comparison table.
8. Show the selected model: `{best_model_name}`.
9. Show the accuracy: `{accuracy:.6f}`.
10. Show that `accuracy.py` outputs only the accuracy number.
11. Show the generated final ZIP: `{SUBMISSION_ZIP_NAME}`.
"""

(OUTPUT_DIR / 'extract_features.py').write_text(extract_script, encoding='utf-8')
(OUTPUT_DIR / 'train_model.py').write_text(train_script, encoding='utf-8')
(OUTPUT_DIR / 'accuracy.py').write_text(accuracy_script, encoding='utf-8')
(OUTPUT_DIR / 'README.md').write_text(readme_text, encoding='utf-8')
(OUTPUT_DIR / 'VIDEO_DEMO_CHECKLIST.md').write_text(video_checklist, encoding='utf-8')

try:
    package_list = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
except Exception as exc:
    package_list = f'Could not generate pip freeze output: {exc}\n'
(OUTPUT_DIR / 'package_list.txt').write_text(package_list, encoding='utf-8')

print('Generated deliverable files:')
for filename in ['extract_features.py', 'train_model.py', 'accuracy.py', 'README.md', 'VIDEO_DEMO_CHECKLIST.md', 'package_list.txt']:
    print(f'  - {OUTPUT_DIR / filename}')

print('\naccuracy.py output should be only one number:')
print((OUTPUT_DIR / 'accuracy.txt').read_text().strip())

## 10. Build the final submission ZIP

This cell packages the final submission ZIP while excluding the auxiliary files you did not want included. The ZIP is saved inside Colab at `/content/output_output_lab3.zip`.



In [ ]:
# ============================================================
# 10. Build the final submission ZIP
# ============================================================
# Per your preference, the files listed below are NOT included in the ZIP.
# They may still exist in /content/malware_detection_outputs/ for inspection,
# but they are skipped during packaging.

EXCLUDED_FROM_ZIP = {
    'extract_features.py',
    'train_model.py',
    'accuracy.py',
    'accuracy.txt',
    'feature_names.json',
    'VIDEO_DEMO_CHECKLIST.md',
    'package_list.txt',
    'full package_list.txt',
}

if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
submission_outputs = SUBMISSION_DIR / 'outputs'
submission_outputs.mkdir(parents=True, exist_ok=True)

skipped_files = []

for path in OUTPUT_DIR.iterdir():
    if not path.is_file():
        continue

    if path.name in EXCLUDED_FROM_ZIP:
        skipped_files.append(path.name)
        continue

    if path.suffix in {'.py', '.md'}:
        shutil.copy2(path, SUBMISSION_DIR / path.name)
    else:
        shutil.copy2(path, submission_outputs / path.name)

if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

with zipfile.ZipFile(SUBMISSION_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(SUBMISSION_DIR.rglob('*')):
        if file_path.is_file():
            arcname = file_path.relative_to(SUBMISSION_DIR)
            if Path(arcname).name in EXCLUDED_FROM_ZIP:
                skipped_files.append(str(arcname))
                continue
            zf.write(file_path, arcname=arcname)

# Safety check: fail loudly if any excluded file accidentally enters the ZIP.
with zipfile.ZipFile(SUBMISSION_ZIP, 'r') as zf:
    zip_contents = zf.namelist()

forbidden_found = [name for name in zip_contents if Path(name).name in EXCLUDED_FROM_ZIP]
if forbidden_found:
    raise RuntimeError(f'Excluded files were found in the ZIP: {forbidden_found}')

print(f'Final submission ZIP created: {SUBMISSION_ZIP}')
print(f'ZIP size: {SUBMISSION_ZIP.stat().st_size / 1024:.1f} KB')
print('\nExcluded from ZIP:')
for name in sorted(set(skipped_files)):
    print(f'  - {name}')

print('\nZIP contents:')
for name in zip_contents:
    print(f'  - {name}')


## 11. Final output paths — no automatic browser download

This final cell does not push anything to your browser. It only prints where the files were saved inside the Colab runtime.

To view the files in Colab, open the left sidebar, click the folder icon, and browse to `/content/`.


In [ ]:
# ============================================================
# 12. Final output paths only; no browser download
# ============================================================
# This cell intentionally does not trigger any browser download.
# All files remain inside the Colab runtime unless you manually move or copy them.

print('Notebook finished successfully.')
print('No automatic browser download was triggered.')
print()
print(f'Final submission ZIP: {SUBMISSION_ZIP}')
print(f'Unpacked submission folder: {SUBMISSION_DIR}')
print(f'Intermediate output folder: {OUTPUT_DIR}')
print()
print('To inspect files in Colab:')
print('  1. Open the left sidebar.')
print('  2. Click the folder icon.')
print('  3. Open /content/.')
print('  4. Find output_output_lab3.zip and malware_detection_outputs/.')
print('  5. The ZIP excludes the auxiliary files listed in the packaging cell.')


## Done

The notebook has completed the full workflow and saved the final ZIP inside the Colab runtime. No automatic browser download is performed.
